# A2A 协议（Agent-to-Agent）· Agent ↔ Agent 的开放通信标准

这一课讲 **A2A**：让「用不同框架、由不同团队/厂商写的智能体」能互相发现、互相下单、互相交货。
一句话定位 ——

> A2A 之于「Agent ↔ Agent」，就等于 ACP 之于「编辑器 ↔ Agent」、MCP 之于「Agent ↔ 工具」。
> 它基于 **JSON-RPC 2.0 over HTTP**：服务端挂一张「智能体名片」（Agent Card）让别人发现，
> 再在同一个地址收 `message/send` 任务。于是 **CrewAI 写的分析师**和 **DeepAgents 写的调度员**
> 两家互不认识的框架，就能用同一套协议串成一条链路。

## 先分清三种协议（本课最常混的地方）

| 协议 | 谁 ↔ 谁 | 解决什么 | 本仓对应 |
|---|---|---|---|
| **ACP** | 编辑器 ↔ 智能体 | 让 Agent 长在 IDE 界面里（LSP 式） | `07_protocols/01_ACP协议.ipynb` |
| **A2A** | **Agent ↔ Agent** | 多个 Agent 互相派活、跨框架协作 | **本课** |
| **MCP** | Agent ↔ 工具/服务 | 让 Agent 用上外部工具、数据 | `Agent/05_mcp/` |

三者的传输层也不同：ACP 走 **stdio**（一行一个 JSON 对象），A2A 走 **HTTP**（POST 一个 JSON），
MCP 两者皆可。但**底层报文全是 JSON-RPC 2.0** —— 所以上一课把 JSON-RPC 看懂之后，
本课就是「换到 HTTP 上、再加一张 Agent Card 做服务发现」。

## 本课合并了五个源文件（⚠️ 编号有错位，别按编号硬配对）

| 源文件 | 角色 | 在本文里的位置 |
|---|---|---|
| `02_a2a服务端.py` | 课案**原版** · 服务端（造一个「时间助手」交给 a2a_auto_wrapper） | 第 1.1 节 |
| `03_a2a服务端_jxsd.py` | 课案**完整版** · 服务端（把 CrewAI 分析师发布成 A2A，含降级真 HTTP） | 第 2 节 |
| `03_a2a客户端.py` | 课案**原版** · 客户端（a2a-sdk 发现 + 调用的伪代码） | 第 1.2 节 |
| `04_a2a客户端_jxsd.py` | 课案**完整版** · 客户端（DeepAgents 总调度员） | 第 3 节 |
| `05_a2a互相通信_jxsd.py` | 课案**完整版** · 互相通信（三方串联，无原版） | 第 4 节 |

编号错位关系：`02_a2a服务端.py` ↔ `03_a2a服务端_jxsd.py`（服务端原版/完整版配对）、
`03_a2a客户端.py` ↔ `04_a2a客户端_jxsd.py`（客户端原版/完整版配对）、`05_a2a互相通信_jxsd.py` 无原版。

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 课案完整流程要 CrewAI 分析师 + DeepAgents 调度员两个服务互调 |
| 依赖（已装） | `langchain` / `langgraph` / `langgraph_sdk` / `deepagents` / `langgraph_cli` |
| 依赖（缺） | **未装**：`crewai` + `a2a_auto_wrapper` —— 只有「真的 serve_crewai_agent」才需要它俩 |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（根目录 `.env`，已配置；**降级路径不调模型**） |
| 前置服务 | 无。**降级演示自起一个纯标准库 HTTP 服务**（`http.server`，随机端口，用完即关） |
| 预计耗时 | 约 5 秒（降级路径，不调模型、不起常驻服务） |

### 想走完整真流程，要装什么 + 起什么

```powershell
uv add a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai
uv add "langgraph-cli[inmem]"    # 提供 langgraph dev 命令
```

然后按顺序起两个服务（**顺序不能反**，原因见第 4 节）：

```text
① .venv\Scripts\python crewai_a2a_server.py          # CrewAI 分析师，监听 2025
② langgraph dev --port 2024 --no-browser              # 总调度员，监听 2024
③ .venv\Scripts\python a2a_client.py                  # 客户端提问
```

> ⚠️ 本课内置**降级**：`crewai` / `a2a_auto_wrapper` 没装时，源文件用 `try/except ImportError`
> 兜住，改走「纯标准库 HTTP 服务 + 假 client」把 A2A 的传输层坐实 —— 不需要网络、不需要模型、
> 不起常驻服务即可跑通全课。本机实测就是走这条降级路径。

## 本节地图

先看「一次提问在三个进程之间怎么走」的全景图，这是 A2A 最值得记住的一张图：

```mermaid
graph LR
    A["a2a_client.py<br/>普通 Python 脚本"] -->|"langgraph_sdk / HTTP"| B["DeepAgents 总调度员<br/>:2024"]
    B -->|"A2AClient / HTTP JSON-RPC"| C["CrewAI 数据分析师<br/>:2025"]
    C -->|"分析报告原路返回"| B
    B -->|"润色后的最终回复"| A
```

**等价文本（裸 JupyterLab 不渲染 mermaid，看这里）**：

```text
a2a_client ──langgraph_sdk（HTTP）──► :2024 总调度员（DeepAgents）
总调度员判断「这题要数据分析」→ 选 call_crewai_analyst 工具
工具内部 ──A2AClient.send_message（HTTP JSON-RPC）──► :2025 CrewAI 分析师
分析师报告原路返回 → 调度员润色成最终回复 → a2a_client 打印
```

再看「客户端怎么发现并调用一个 A2A 服务」的发现流程（本课第 2、3 节会真的跑一遍）：

```mermaid
sequenceDiagram
    autonumber
    participant C as 客户端（A2AClient）
    participant S as A2A 服务端（:2025）
    C->>S: GET /.well-known/agent.json
    S-->>C: Agent Card（name / skills / url / capabilities…）
    C->>S: POST /  message/send（JSON-RPC，body 是任务消息）
    S-->>C: result（taskId / status.state / artifacts[].parts[].text）
```

**等价文本（裸 JupyterLab 不渲染 mermaid）**：

```text
→ GET /.well-known/agent.json                              拉名片：我是谁、会什么、地址在哪
← {"name":"CrewAI 数据分析师","skills":[...],"url":"..."}
→ POST /  {"jsonrpc":"2.0","id":1,"method":"message/send","params":{"message":{...}}}
← {"jsonrpc":"2.0","id":1,"result":{"id":taskId,"status":{"state":"completed"},"artifacts":[...]}}
```

### 与上下节的衔接

- **上一课**（`01_ACP协议.ipynb`）：JSON-RPC 2.0 的收发 —— 本课把它搬到 HTTP 上，再加 Agent Card。
- **本课**：A2A 的三个端点 —— 服务端（CrewAI 分析师）、客户端（DeepAgents 调度员）、三方互相通信。
- **MCP**（`05_mcp/`）：Agent ↔ 工具，与本课的 Agent ↔ Agent 互不替代。

## 0. 环境引导

notebook 的工作目录默认是**它自己所在的文件夹**（`Agent/07_protocols/`），
而本项目所有代码都写 `from config import settings`（`config.py` 在仓库根）。

所以第一格统一做一件事：**向上找到仓库根 → `chdir` 过去 → 塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 0.1 前置自检：本课哪些依赖在、哪些缺

这是本课的**第一个教学点**：A2A 的完整真流程需要两个「重」依赖 ——
`crewai`（造 CrewAI Agent）与 `a2a_auto_wrapper`（`serve_crewai_agent` / `A2AClient`）。
本机都没装，所以全程走降级演示。下面先自检一遍，把「缺什么」钉死。

In [ ]:
import importlib.util


def _has_module(name: str) -> bool:
    """探测某个包是否可 import（不真正导入，只看能否定位到）。"""
    return importlib.util.find_spec(name) is not None


for _name in ("crewai", "a2a_auto_wrapper", "langgraph_sdk", "deepagents", "langgraph_cli"):
    print(f"  {_name:<20}", "已装" if _has_module(_name) else "未装")

print()
print("结论：crewai / a2a_auto_wrapper 未装 → 本课走「降级演示」（纯标准库 HTTP 服务 + 假 client）。")
print("      这不需要网络、不需要模型，全程离线可跑；完整真流程的安装命令见「运行条件」。")

### 预期输出

```text
  crewai               未装
  a2a_auto_wrapper     未装
  langgraph_sdk        已装
  deepagents           已装
  langgraph_cli        已装

结论：crewai / a2a_auto_wrapper 未装 → 本课走「降级演示」（纯标准库 HTTP 服务 + 假 client）。
      这不需要网络、不需要模型，全程离线可跑；完整真流程的安装命令见「运行条件」。
```

## 1. 课案原版：最短实现（服务端 + 客户端）

课案原版只有两个短文件：服务端 55 行、客户端 56 行。它们**都不真正起服务**，
只把「A2A 的两个端点长什么样」摆出来 —— 服务端负责「造 Agent 并交给 a2a_auto_wrapper 发布」，
客户端负责「用 a2a-sdk 发现并调用」。先看最短实现，再看第 2、3 节的完整版。

### 1.1 服务端原版（`02_a2a服务端.py`，55 行）

原版服务端只做一件事：**造一个 LangChain Agent（时间助手）**，然后交给
`a2a_auto_wrapper` 一行包装发布。注意它**没有自己起服务** —— 原脚本末尾的 `print`
明确写着「本文件由 a2a_auto_wrapper 包装发布（不要直接执行）」。

- `init_chat_model(model_provider="openai", ...)`：用「OpenAI 兼容」这一档接本项目 `.env` 的模型网关；
- `create_agent(model=llm, tools=[get_time], system_prompt=...)`：造一个带 `get_time` 工具的 Agent；
- 关键约定：模块暴露一个 **`agent` 对象**（`a2a_auto_wrapper` 会去找它并发布）。

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


@tool
def get_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# a2a_auto_wrapper 会寻找 `agent` 对象并发布
agent = create_agent(
    model=llm,
    tools=[get_time],
    system_prompt="你是时间助手，回答时间问题请使用工具。",
)

原版末尾（原脚本 `if __name__ == "__main__":` 里的三行）只是打印发布提示，
不调用模型、不起服务：

In [ ]:
print("本文件由 a2a_auto_wrapper 包装发布（不要直接执行）")
print("发布命令：a2a-auto-wrap --module 07_protocols.02_a2a服务端 --url http://localhost:10000")
print("发布后可访问 http://localhost:10000/.well-known/agent.json 查看名片")

### 预期输出

```text
本文件由 a2a_auto_wrapper 包装发布（不要直接执行）
发布命令：a2a-auto-wrap --module 07_protocols.02_a2a服务端 --url http://localhost:10000
发布后可访问 http://localhost:10000/.well-known/agent.json 查看名片
```

**注意这个反直觉的地方**：这一格看起来「什么都没干」—— 没有 `.invoke()`、没有回复。
它只是把 `agent` 造出来放在模块命名空间里，等 `a2a_auto_wrapper` 来取。真正的「运行方式」是：

```text
a2a-auto-wrap → import 这个模块 → 拿走 agent → 发布成 A2A 服务（挂名片 + 收任务）
```

### 1.2 客户端原版（`03_a2a客户端.py`，56 行）

原版客户端是一段**伪代码**，用官方 SDK `a2a-sdk`（`A2ACardResolver` / `A2AClient`）
演示两步：先 `get_agent_card()` 拉名片（发现），再 `send_message(...)` 发任务（调用）。
原脚本只是把这段伪代码 `print` 出来，不会真的执行（因为 `a2a-sdk` 也没装）。

In [ ]:
AGENT_URL = "http://localhost:10000"

print(
    """
# ---------- A2A 客户端伪代码 ----------
from a2a.client import A2ACardResolver, A2AClient
import httpx

async def main():
    async with httpx.AsyncClient() as httpx_client:
        # 1. 拉取远程 Agent 的「名片」（能力发现）
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=AGENT_URL)
        card = await resolver.get_agent_card()
        print("发现智能体：", card.name, card.description)

        # 2. 创建客户端并发送任务
        client = A2AClient(httpx_client=httpx_client, agent_card=card)
        response = await client.send_message(
            {
                "message": {
                    "role": "user",
                    "parts": [{"type": "text", "text": "现在几点了？"}],
                }
            }
        )
        print("远程 Agent 回复：", response)

# A2A 的意义：不管对端是 LangGraph 还是 CrewAI 写的 Agent，
# 只要它说 A2A 协议，你都能用同样方式发现并调用它。
"""
)

### 预期输出

```text

# ---------- A2A 客户端伪代码 ----------
from a2a.client import A2ACardResolver, A2AClient
import httpx

async def main():
    async with httpx.AsyncClient() as httpx_client:
        # 1. 拉取远程 Agent 的「名片」（能力发现）
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=AGENT_URL)
        card = await resolver.get_agent_card()
        print("发现智能体：", card.name, card.description)

        # 2. 创建客户端并发送任务
        client = A2AClient(httpx_client=httpx_client, agent_card=card)
        response = await client.send_message(
            {
                "message": {
                    "role": "user",
                    "parts": [{"type": "text", "text": "现在几点了？"}],
                }
            }
        )
        print("远程 Agent 回复：", response)

# A2A 的意义：不管对端是 LangGraph 还是 CrewAI 写的 Agent，
# 只要它说 A2A 协议，你都能用同样方式发现并调用它。
```

**两件事值得记住**：① A2A 客户端永远是「先 `get_agent_card` 拉名片、再 `send_message` 发任务」两步；
② 伪代码里 `parts: [{"type": "text", ...}]` 是**内容块数组**，文本/文件/结构化数据都能塞。

## 2. 完整版：服务端 —— 把 CrewAI 分析师发布成 A2A 服务

原版把「造 Agent」和「发布」拆开了，但**「发布」那一层被 a2a_auto_wrapper 藏掉了**。
完整版把它补全，并诚实演示「本机没装 crewai / a2a_auto_wrapper 时怎么办」：

- **真流程**：`serve_crewai_agent(analyst, port=2025)` 一行把 CrewAI Agent 暴露成 A2A 服务；
- **降级演示**：用**纯标准库**起一个真的 HTTP JSON-RPC 服务（随机端口、演示完自动关），
  让 `urllib` 真的把 `message/send` 发过去、真的收回来 —— 「A2A = JSON-RPC 2.0 over HTTP」就不再是空话。

本机走的是降级路径。

### 2.1 依赖探测：crewai / a2a_auto_wrapper 都没装

按本仓「**模块层面永远不许因为缺包而 import 失败**」的铁律，两个第三方 import 都用
`try/except ImportError` 兜住，把「缺什么」记在布尔开关里，等到真正要跑的时候再决定
走真流程还是降级演示。

In [ ]:
import json
import threading
import urllib.error
import urllib.request
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from config import settings

_HAS_CREWAI = False
_HAS_A2A_WRAPPER = False
_IMPORT_ERRORS: list[str] = []

try:
    from crewai import Agent, LLM  # type: ignore[import-not-found]

    _HAS_CREWAI = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"crewai：{exc}")

try:
    from a2a_auto_wrapper import serve_crewai_agent  # type: ignore[import-not-found]

    _HAS_A2A_WRAPPER = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"a2a_auto_wrapper：{exc}")

# 课案里的端口：CrewAI 分析师固定监听 2025
CREWAI_A2A_PORT = 2025

### 2.2 课案原文 `crewai_a2a_server.py`（29 行）+ `build_analyst`

课案里那个 CrewAI 服务端只有 29 行。下面把它存成字符串常量（供 `main()` 逐行打印对照），
并把「造分析师」抽成 `build_analyst()`。CrewAI 的心智模型是「招一个员工」：
`role`（职位）/ `goal`（目标）/ `backstory`（履历）三段式。

课案到本项目的两处改写（规范第 3 节）：

1. `from config import setting` → `from config import settings`；
2. `setting.MODEL_NAME` 这种全大写下划线 → 本项目的 `settings.model_name`（`.env` 字段是小写）。

In [ ]:
COURSE_SERVER_PY = '''"""CrewAI A2A 服务 — python crewai_a2a_server.py 启动（监听 2025 端口）"""
from crewai import Agent, LLM
from config import settings                 # 课案原文是 from config import setting


# 数据分析师 Agent：作为 A2A 服务端，等待 DeepAgents 调度员调用
analyst = Agent(
    role="数据分析师",
    goal="对数据进行深度分析并生成结构化报告",
    backstory="你是资深数据分析师，擅长数据洞察和趋势预测。",
    llm=LLM(
        model=settings.model_name,          # 课案原文 setting.MODEL_NAME
        provider="openai",
        base_url=settings.base_url,         # 课案原文 setting.BASE_URL
        api_key=settings.api_key,           # 课案原文 setting.API_KEY
    ),
)


if __name__ == "__main__":
    from a2a_auto_wrapper import serve_crewai_agent

    # 一行代码将 CrewAI Agent 暴露为 A2A 服务：http://localhost:2025
    serve_crewai_agent(analyst, port=2025)
'''


def build_analyst():
    """按课案原文构造 CrewAI 数据分析师——只在 crewai 装好时才会被调用。

    这里刻意保持课案的三段式（role / goal / backstory），
    因为 CrewAI 的心智模型就是「招一个员工」：给它职位、目标、履历。
    """
    return Agent(
        role="数据分析师",          # 职位：决定它「是谁」
        goal="对数据进行深度分析并生成结构化报告",   # 目标：决定它「要达成什么」
        backstory="你是资深数据分析师，擅长数据洞察和趋势预测。",   # 履历：决定它「怎么想问题」
        # LLM 的三件套全部来自 settings（.env），一个都不许硬编码
        llm=LLM(
            model=settings.model_name,
            provider="openai",
            base_url=settings.base_url,
            api_key=settings.api_key,
        ),
    )

### 为什么这一格没有输出

本格只做了两次赋值（`COURSE_SERVER_PY` 字符串 + `build_analyst` 函数定义），**没有输出**。
注意 `build_analyst()` 的**函数体**里用了 `Agent(...)` / `LLM(...)` —— 这两个名字来自
第 2.1 小节那个**失败的** `from crewai import ...`，所以现在并不存在；但只要不调用
`build_analyst()`（真流程才调用），就不会 `NameError`。打印样子见第 2.7 小节 `main()` 的输出。

### 2.3 Agent Card（智能体名片）：A2A 的第一块拼图

A2A 和 ACP 最大的不同：**对方在哪、会什么，要能被「发现」**。所以 A2A 规定每个服务必须
在固定路径上挂一张名片：

```text
GET /.well-known/agent.json        （老版本，2025 年前后的教程都写这个）
GET /.well-known/agent-card.json   （新版官方路径，两者内容一致）
```

名片字段逐个解释：

| 字段 | 说明 |
|---|---|
| `name` / `description` / `version` | 自我介绍，客户端拿它渲染「我发现了哪个智能体」 |
| `url` | **A2A 服务的基础地址**，后面所有 JSON-RPC 都往这里 POST |
| `protocolVersion` | A2A 协议版本，双方不一致就别硬聊 |
| `preferredTransport` | JSONRPC / GRPC / HTTP+JSON，多数实现是 JSONRPC |
| `capabilities.streaming` | 是否支持流式（对应 `message/stream` 方法） |
| `capabilities.pushNotifications` | 是否支持服务端主动回调推送 |
| `defaultInputModes` / `defaultOutputModes` | 收发的内容形态，一般是 `["text/plain"]` |
| `skills[]` | **能力清单**：会干什么、给样例、有哪些标签 |
| `skills[].id` | 技能唯一标识；官方建议用「url + 技能名」算 UUID，避免撞名 |
| `skills[].examples` | 自然语言示例，调度员（LLM）主要靠它决定要不要调用 |

注意 `skills[].id` 用 `uuid.uuid5(NAMESPACE_URL, f"{base_url}#{skill_name}")`：
同一台服务、同一个技能名**永远算出同一个 id**（确定性哈希），不用持久化。

In [ ]:
def build_agent_card(base_url: str) -> dict:
    """生成 CrewAI 分析师的 Agent Card。"""
    skill_name = "数据分析与趋势预测"
    return {
        # ---- 自我介绍：客户端拿这几项渲染「我发现了哪个智能体」 ----
        "name": "CrewAI 数据分析师",
        "description": "对数据进行深度分析并生成结构化报告，擅长趋势预测。",
        # uuid5：同一台服务、同一个技能名永远算出同一个 id（确定性哈希）
        "url": base_url,            # ★ A2A 服务的基础地址，后续所有 JSON-RPC 都往这里 POST
        "version": "1.0.0",         # 这个 Agent 自己的版本号（不是协议版本）
        # ---- 协议协商：双方版本不一致就别硬聊 ----
        "protocolVersion": "0.3.0",
        "preferredTransport": "JSONRPC",     # 也可 GRPC / HTTP+JSON，多数实现是 JSONRPC
        # ---- 能力位：客户端据此决定「能不能用某个方法」 ----
        "capabilities": {
            "streaming": True,       # 支持 message/stream（流式返回）
            "pushNotifications": False,   # 不支持服务端主动回调推送
        },
        "defaultInputModes": ["text/plain"],    # 我能收什么形态
        "defaultOutputModes": ["text/plain"],   # 我会吐什么形态
        # ---- skills[]：能力清单，调度员（LLM）主要靠这一节决定要不要调用你 ----
        "skills": [
            {
                # 技能唯一标识；官方建议用 url + 技能名 算 UUID，避免不同 Agent 之间撞名
                "id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"{base_url}#{skill_name}")),
                "name": skill_name,
                "description": "接收一段业务数据描述，输出趋势判断、异常点与建议。",
                "tags": ["数据分析", "趋势预测", "报告生成"],
                # examples 是**自然语言示例**：模型判断「这个活像不像它举的例子」，比 tags 更管用
                "examples": ["分析销售数据：1月100万 2月120万 3月90万，做趋势预测"],
            }
        ],
    }

### 2.4 任务消息 `message/send`：A2A 的第二块拼图

消息体和 ACP 的 `session/prompt` 神似，都是「角色 + 内容块数组」：

```json
{"jsonrpc":"2.0","id":1,"method":"message/send",
 "params":{"message":{"role":"user","parts":[{"kind":"text","text":"分析销售数据…"}],"messageId":"<客户端生成，用于去重>"}}}
```

响应把「任务」整体返回，关键字段：

| 字段 | 说明 |
|---|---|
| `result.id` | taskId，后续用 `tasks/get` 查进度 |
| `result.status.state` | `submitted` / `working` / `completed` / `failed` / `canceled` |
| `result.artifacts[].parts[].text` | **Agent 的产出正文就在这里** |
| `result.history[]` | 往返消息历史 |

方法名演进（不同版本教程会打架，认准你装的包）：

| 版本 | 发消息 | 查任务 |
|---|---|---|
| 0.2.x | `tasks/send` | `tasks/get` |
| 0.3.x | `message/send` | `tasks/get` |
| — | `message/stream`（流式，配 `capabilities.streaming=true`） | — |

In [ ]:
def build_task_message(query: str) -> dict:
    """构造一条 A2A 的 message/send 请求。"""
    return {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "message/send",
        "params": {
            "message": {
                # role 只能是 user / agent —— 和 OpenAI messages 的角色含义一样
                "role": "user",
                # parts 是内容块数组：文本、文件、结构化数据都能塞
                "parts": [{"kind": "text", "text": query}],
                "messageId": str(uuid.uuid4()),
            }
        },
    }

### 2.5 降级演示用的「真 HTTP」A2A 服务（纯标准库，零依赖）

目的不是替代 `a2a_auto_wrapper`，而是用几十行把 A2A 的**传输层**坐实：

- 名片：`GET /.well-known/agent.json`
- 调用：`POST /`（body 是 JSON-RPC，`Content-Type: application/json`）

真实实现（`serve_crewai_agent` / AMP Factory）也是这两个端点，只是内部接了 LLM。
降级版不调 LLM，直接回一段**写死的报告**，但**报文结构是真的**。

In [ ]:
FAKE_ANALYST_REPORT = (
    "【趋势判断】1→2 月增长 20%，2→3 月回落 25%，属单峰波动，非持续增长。\n"
    "【异常点】3 月环比 -25% 需排查（促销结束？渠道断货？）。\n"
    "【建议】把 2 月作为活动基准，Q2 目标按 1 月水平 +10% 设定更稳。"
)


class _A2AHandler(BaseHTTPRequestHandler):
    """最小 A2A 端点实现：一个 GET 名片 + 一个 POST JSON-RPC。"""

    # 让 handler 能拿到外面构造好的名片
    agent_card: dict = {}

    def log_message(self, fmt, *args):  # 关掉 http.server 默认的 stderr 噪音
        return

    def _send_json(self, payload, status: int = 200) -> None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self) -> None:  # noqa: N802 - BaseHTTPRequestHandler 规定的方法名
        if self.path in ("/.well-known/agent.json", "/.well-known/agent-card.json"):
            # 能力发现：客户端启动时第一件事就是拉这张名片
            self._send_json(self.agent_card)
        else:
            self._send_json({"error": "not found", "path": self.path}, status=404)

    def do_POST(self) -> None:  # noqa: N802
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length).decode("utf-8")
        try:
            request = json.loads(raw)
        except json.JSONDecodeError:
            # JSON-RPC 标准错误码 -32700
            self._send_json(
                {"jsonrpc": "2.0", "id": None,
                 "error": {"code": -32700, "message": "Parse error"}}
            )
            return

        method = request.get("method")
        req_id = request.get("id")

        if method == "message/send":
            message = (request.get("params") or {}).get("message") or {}
            # 取出用户问的文本（parts 里 kind == "text" 的那些拼起来）
            # —— A2A 的 parts 是**内容块数组**，所以要过滤 + 拼接，不能直接当字符串用
            query = " ".join(
                p.get("text", "") for p in message.get("parts", []) if p.get("kind") == "text"
            )
            # 真实服务端这里会把 query 交给 CrewAI Agent 跑；
            # 降级版直接回一段写死的报告，但**报文结构是真的**。
            self._send_json(
                {
                    "jsonrpc": "2.0",
                    "id": req_id,       # ★ 必须回同 id：这是 JSON-RPC 的配对规则
                    "result": {
                        "id": str(uuid.uuid4()),          # taskId
                        "contextId": str(uuid.uuid4()),   # 会话上下文 id
                        # status.state 是任务的生命周期：
                        # submitted（已接收）→ working（处理中）→ completed / failed / canceled
                        # 长任务用 message/send 拿到 taskId 后，可以再 tasks/get 轮询进度
                        "status": {"state": "completed", "timestamp": "2026-01-01T00:00:00Z"},
                        # ★ 产出正文放在 artifacts 里（不是放在 status 里）——
                        #   一个任务可以产出多份 artifact，每份又由多个 part 组成
                        "artifacts": [
                            {
                                "artifactId": str(uuid.uuid4()),
                                "name": "分析报告",     # 给客户端展示用的可读名字
                                "parts": [
                                    {
                                        # kind="text" 和 A2A 请求里的 parts 对齐；
                                        # 换成 "file"/"data" 就能回文件或结构化数据
                                        "kind": "text",
                                        "text": f"[降级演示·非真实 CrewAI 输出]\n收到问题：{query}\n\n"
                                                + FAKE_ANALYST_REPORT,
                                    }
                                ],
                            }
                        ],
                        # history 原样带回往返消息，方便客户端做审计/多轮拼接
                        "history": [message],
                    },
                }
            )
            return

        # tasks/get：按 taskId 查一个**已经提交过**的任务的当前状态。
        # 这里简化成「直接回答 completed」，真实实现要维护一张 taskId → 状态 的表。
        if method == "tasks/get":
            self._send_json(
                {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "result": {
                        "id": (request.get("params") or {}).get("id"),
                        "status": {"state": "completed"},
                    },
                }
            )
            return

        # 方法不认识 → JSON-RPC -32601
        self._send_json(
            {"jsonrpc": "2.0", "id": req_id,
             "error": {"code": -32601, "message": f"Method not found: {method}"}}
        )

### 2.6 起服务 + 收发 + 安装提示

`start_fake_a2a_server()` 用 `ThreadingHTTPServer(("127.0.0.1", 0), ...)` —— **端口 0 = 让
操作系统分配一个空闲端口**（课案固定 2025 是「真部署」才需要的稳定性；演示用随机端口才不会跟
别的进程抢，对并发改造的 notebook 也更安全）。服务跑在**守护线程**里，`serve_forever` 不会阻塞内核。

`http_get_json` / `http_post_json` 用标准库 `urllib` 真的发 HTTP；`print_install_hint` 把
「缺的是哪个模块」原样打出来，学员能直接对着装。

In [ ]:
def start_fake_a2a_server() -> tuple[ThreadingHTTPServer, str]:
    """起一个真的 HTTP 服务（端口 0 = 让系统分配空闲端口，避免撞车）。

    返回 (server, base_url)。跑完记得 server.shutdown()。
    """
    # port=0：让操作系统给个空闲端口。课案用的是固定 2025，
    # 但那是「真部署」才需要的稳定性；演示里用随机端口才不会跟别的进程抢。
    server = ThreadingHTTPServer(("127.0.0.1", 0), _A2AHandler)
    base_url = f"http://127.0.0.1:{server.server_address[1]}"
    _A2AHandler.agent_card = build_agent_card(base_url)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, base_url


def http_get_json(url: str) -> dict:
    """GET 一个 JSON 端点（能力发现走这里）。"""
    # noqa: S310 —— 目标是本地回环地址，不是用户可控的 URL
    with urllib.request.urlopen(url, timeout=10) as resp:  # noqa: S310 - 本地回环地址
        return json.loads(resp.read().decode("utf-8"))


def http_post_json(url: str, payload: dict) -> dict:
    """POST 一个 JSON-RPC 报文（真正的调用走这里）。"""
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    # Content-Type 必须是 application/json —— A2A 靠它区分「这是协议报文」而不是普通表单
    req = urllib.request.Request(
        url,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    # noqa: S310 —— 同上，本地回环
    with urllib.request.urlopen(req, timeout=10) as resp:  # noqa: S310 - 本地回环地址
        return json.loads(resp.read().decode("utf-8"))


def print_install_hint() -> None:
    print("【前置条件检查】本机 venv 缺少 A2A 相关依赖：")
    for err in _IMPORT_ERRORS:
        # 原始 ImportError 文本里带着模块名，方便对号入座
        print(f"    - {err}")
    print()
    print("课案安装命令（本项目统一用 uv）：")
    print("    uv add a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai")
    print("    # 或：F:\\ProGram\\Python_Base\\.venv\\Scripts\\python.exe -m pip install "
          "a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai")
    print("    # 注：crewai 体积不小（会带 litellm / embedchain 等一大串依赖），装前有个心理准备")
    print()

### 2.7 主流程 `main()`：真跑一遍「发现 → 调用 → 错误路径 → 关服务」

`main()` 把上面拼起来，降级路径做三件事：① 拉名片、② 发 `message/send` 收报告、③ 故意写错方法名看 `-32601`。
全程在 `try/finally` 里，`finally` 里 `server.shutdown()` 保证**无论成功失败都把服务关掉**。

In [ ]:
def main() -> None:
    print("=" * 66)
    print("A2A 服务端：把 CrewAI 数据分析师发布为 A2A 服务（课案端口 2025）")
    print("=" * 66)
    print()

    # ---------- 第 1 步：课案原文 ----------
    print("【课案原文】crewai_a2a_server.py（29 行）：")
    for line in COURSE_SERVER_PY.splitlines():
        print("    " + line)
    print()
    print("【启动命令（课案原文）】")
    print("    $env:PYTHONUTF8='1'; .venv\\Scripts\\python crewai_a2a_server.py")
    print("    # $env:PYTHONUTF8='1' 是为了让 CrewAI 打印中文日志时不乱码")
    print()

    # ---------- 第 2 步：依赖情况 ----------
    print_install_hint()

    if _HAS_CREWAI and _HAS_A2A_WRAPPER:
        print("【依赖检查】crewai + a2a_auto_wrapper 均已就绪，按课案原文启动真实 A2A 服务。")
        print(f"            监听端口 {CREWAI_A2A_PORT}，Ctrl+C 退出。")
        print(f"            名片地址：http://localhost:{CREWAI_A2A_PORT}/.well-known/agent.json")
        print()
        analyst = build_analyst()
        try:
            # 课案原话：一行代码将 CrewAI Agent 暴露为 A2A 服务
            serve_crewai_agent(analyst, port=CREWAI_A2A_PORT)
        except KeyboardInterrupt:
            print("\n已手动停止 A2A 服务。")
        return

    # ---------- 第 3 步：降级演示 —— 先看名片 ----------
    print("【降级演示】未安装 crewai / a2a_auto_wrapper，无法真的 serve_crewai_agent(analyst, port=2025)。")
    print("            改用纯标准库起一个真 HTTP 服务，把 A2A 的传输层坐实。")
    print()
    server, base_url = start_fake_a2a_server()
    print(f"已启动降级版 A2A 服务：{base_url}（端口由系统分配，演示完自动关闭）")
    print()

    try:
        # ① 能力发现：真实 A2A 客户端第一步也是拉这张名片，才知道对方会什么、地址在哪
        card = http_get_json(f"{base_url}/.well-known/agent.json")
        print("① 能力发现：GET /.well-known/agent.json → Agent Card（智能体名片）")
        for line in json.dumps(card, ensure_ascii=False, indent=2).splitlines():
            print("    " + line)
        print()

        query = "分析销售数据：1月100万 2月120万 3月90万，做趋势预测"
        request = build_task_message(query)
        # 打印↔接收共用同一份 request 对象，所以看到的报文就是真正发出去的报文
        print("② 调用：POST / ，body 是 JSON-RPC 的 message/send")
        print("    → " + json.dumps(request, ensure_ascii=False, separators=(",", ":")))
        response = http_post_json(base_url, request)
        # 响应可能很长（artifacts 里塞了整份报告），先截 120 字符给个全貌
        print("    ← " + json.dumps(response, ensure_ascii=False, separators=(",", ":"))[:120] + " …")
        print()
        # 逐字段读响应：这是「怎么从 A2A 响应里把 Agent 的产出抠出来」的标准动作
        result = response["result"]
        print(f"    taskId    = {result['id']}")
        print(f"    state     = {result['status']['state']}")
        print("    产出正文（artifacts[0].parts[0].text）：")
        for line in result["artifacts"][0]["parts"][0]["text"].splitlines():
            print("        " + line)
        print()

        # 顺手演示一下错误路径：方法名写错会拿到 -32601
        # （A2A 复用 JSON-RPC 的标准错误码，和 02_acp原理_jxsd.py 里那套是同一份规范）
        bad = {"jsonrpc": "2.0", "id": 9, "method": "message/foo", "params": {}}
        print("③ 错误路径：方法名写错 → JSON-RPC -32601")
        print("    → " + json.dumps(bad, ensure_ascii=False, separators=(",", ":")))
        print("    ← " + json.dumps(http_post_json(base_url, bad), ensure_ascii=False,
                                   separators=(",", ":")))
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        # 本地回环也可能被代理/防火墙拦掉；协议内容已经打印过，所以这里只提示不中断
        print(f"    本地 HTTP 演示失败（{type(exc).__name__}）：{exc}")
        print("    多为端口/防火墙/代理拦截 127.0.0.1 所致；协议内容仍以上面的报文为准。")
    finally:
        # 无论成功失败都要关服务，否则线程会一直挂着
        server.shutdown()
        server.server_close()
        print()
        print("降级版 A2A 服务已关闭。")
        print()

    # 收口：把「A2A 服务端到底要做哪两件事」压缩成两条，并指向下一节（客户端怎么调它）
    print("=" * 66)
    print("小结：A2A 服务端就两件事 ——")
    print("    ① 在 /.well-known/agent.json 挂一张「名片」（我是谁、会什么、在哪）")
    print("    ② 在同一地址收 JSON-RPC 的 message/send（GET 发现 + POST 调用）")
    print("    serve_crewai_agent(analyst, port=2025) 就是把这两件事自动配好；")
    print("    真实环境下这里跑的是 CrewAI 的 LLM，降级版换成了一段写死的报告。")
    print("    下一步：DeepAgents 调度员怎么调它 → 04_a2a客户端_jxsd.py")
    print("=" * 66)


main()

### 预期输出

本机实测（`crewai` / `a2a_auto_wrapper` 缺 → 走降级分支，真的起了一个随机端口 HTTP 服务又关掉）：

```text
【课案原文】crewai_a2a_server.py（29 行）：
    """CrewAI A2A 服务 — python crewai_a2a_server.py 启动（监听 2025 端口）"""
    from crewai import Agent, LLM
    from config import settings                 # 课案原文是 from config import setting
    ...
    serve_crewai_agent(analyst, port=2025)

【启动命令（课案原文）】
    $env:PYTHONUTF8='1'; .venv\Scripts\python crewai_a2a_server.py
    # $env:PYTHONUTF8='1' 是为了让 CrewAI 打印中文日志时不乱码

【前置条件检查】本机 venv 缺少 A2A 相关依赖：
    - crewai：No module named 'crewai'
    - a2a_auto_wrapper：No module named 'a2a_auto_wrapper'
    ...
【降级演示】未安装 crewai / a2a_auto_wrapper，无法真的 serve_crewai_agent(analyst, port=2025)。
            改用纯标准库起一个真 HTTP 服务，把 A2A 的传输层坐实。

已启动降级版 A2A 服务：http://127.0.0.1:<随机端口>（端口由系统分配，演示完自动关闭）

① 能力发现：GET /.well-known/agent.json → Agent Card（智能体名片）
    { ... "name": "CrewAI 数据分析师", "skills": [...] ... }
② 调用：POST / ，body 是 JSON-RPC 的 message/send
    → {"jsonrpc":"2.0","id":1,"method":"message/send","params":{"message":{"role":"user",...}}}
    ← {"jsonrpc":"2.0","id":1,"result":{"id":"<uuid>","status":{"state":"completed"},...}} …
    taskId    = <uuid>
    state     = completed
    产出正文（artifacts[0].parts[0].text）：
        [降级演示·非真实 CrewAI 输出]
        收到问题：分析销售数据：1月100万 2月120万 3月90万，做趋势预测
        【趋势判断】1→2 月增长 20%，2→3 月回落 25%，属单峰波动，非持续增长。
        【异常点】3 月环比 -25% 需排查（促销结束？渠道断货？）。
        【建议】把 2 月作为活动基准，Q2 目标按 1 月水平 +10% 设定更稳。
③ 错误路径：方法名写错 → JSON-RPC -32601
    → {"jsonrpc":"2.0","id":9,"method":"message/foo","params":{}}
    ← {"jsonrpc":"2.0","id":9,"error":{"code":-32601,"message":"Method not found: message/foo"}}

降级版 A2A 服务已关闭。
```

> ⚠️ 上面的 `<随机端口>` / `<uuid>` 是**每次运行都不同**的随机值（端口由 OS 分配、taskId/messageId
> 是 `uuid4()` 随机生成），只有「结构」是固定的；正文里那几段是实测值。

## 3. 完整版：客户端 —— DeepAgents 总调度员

服务端把「CrewAI 分析师」暴露成 A2A 服务之后，谁来调它？答案是另一个 Agent：**DeepAgents 总调度员**。
本节的通信双方：

- 客户端（本节）→ DeepAgents 总调度员（`A2AClient` + `create_deep_agent`）；
- DeepAgents 总调度员 → CrewAI 数据分析师（上一节那个 :2025 服务）。

关键设计：**调度员把「调另一个 Agent」封装成一个 `@tool`**（`call_crewai_analyst`）——
对 LLM 来说，它和「查天气」「读文件」没有区别，模型只需要在合适时机选这个工具；
工具背后是本地函数还是跨进程、跨框架的 A2A 调用，模型完全不用知道。这就是 A2A 的价值。

### 3.1 依赖探测 + 一个 notebook 专属的 `run_async` 助手

依赖探测和上一节一样用 `try/except ImportError` 兜住。另外这里**多了一个 notebook 专属的适配**：

源脚本里是 `asyncio.run(probe_crewai_tool(...))`，但 Jupyter 内核**本身就跑在一个事件循环里**，
直接 `asyncio.run()` 会抛 `RuntimeError: asyncio.run() cannot be called from a running event loop`。
所以用 `run_async(coro)`：**另起一个线程**去 `asyncio.run()`，再把线程内的异常收集回主线程。
这是本课唯一一处「为了适配 notebook 而改写的代码行」（见交付报告「与源文件的差异」）。

In [ ]:
import asyncio
import platform
from pathlib import Path

from config import settings


def run_async(coro):
    """在独立线程里跑协程：ipykernel 内核已有事件循环，直接 asyncio.run() 会抛
    RuntimeError（cannot be called from a running event loop）。线程内异常收集回主线程。
    """
    box = {}

    def _runner() -> None:
        try:
            box["result"] = asyncio.run(coro)
        except BaseException as exc:  # 把协程里的异常带回主线程
            box["error"] = exc

    t = threading.Thread(target=_runner, daemon=True)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box.get("result")


# | 包               | 作用                                          | 本机 |
# | a2a_auto_wrapper | A2AClient（调度员调 CrewAI 的那只手）          | 缺   |
# | deepagents       | create_deep_agent（总调度员本体）              | 已装 |
# | langgraph_sdk    | 通过 HTTP 调 2024 端口的调度员（05 节要用）     | 已装 |
# | langgraph_cli    | `langgraph dev` 命令本身                       | 缺   |
_HAS_A2A_WRAPPER = False
_HAS_DEEPAGENTS = False
_HAS_LANGGRAPH_SDK = False
_HAS_LANGGRAPH_CLI = False
_IMPORT_ERRORS: list[str] = []

try:
    from a2a_auto_wrapper import A2AClient  # type: ignore[import-not-found]

    _HAS_A2A_WRAPPER = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"a2a_auto_wrapper：{exc}")

try:
    from deepagents import create_deep_agent

    _HAS_DEEPAGENTS = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"deepagents：{exc}")

try:
    from langgraph_sdk import get_client

    _HAS_LANGGRAPH_SDK = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"langgraph_sdk：{exc}")

try:
    from langgraph_cli import cli  # type: ignore[import-not-found]  # noqa: F401

    _HAS_LANGGRAPH_CLI = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"langgraph-cli：{exc}")

# 两个服务的地址（课案固定端口：调度员 2024，CrewAI 分析师 2025）
DEEPAGENT_PORT = 2024
CREWAI_ANALYST_URL = "http://localhost:2025/"

### 3.2 课案原文 `deepagent_a2a.py`（58 行）+ `langgraph.json`

课案里的调度员 58 行，含三处值得抄下来的细节：

| 写法 | 为什么 |
|---|---|
| `@tool` + **async def** | LangGraph/LangChain 支持异步工具；A2A 调用是网络 IO，写成 async 才不堵事件循环 |
| docstring 里的 `Args`/`Returns` | **这就是给 LLM 看的工具说明书**，写得越清楚，模型越会在正确时机调用 |
| try/except 里返回可读提示 | 工具**不要往外抛异常**：抛了会变成 Agent 运行失败；返回一句人能看懂的话，模型还能自己决定重试 |

`langgraph.json` 把 `graph = agent` 这个变量注册成 HTTP 服务上的一张「图」，`deep_researcher` 就是它的键名。

In [ ]:
COURSE_DEEPAGENT_PY = '''"""DeepAgents A2A 服务：总调度员，可自主调用 CrewAI 分析师

通过 langgraph dev --port 2024 暴露为 A2A 服务，加载入口见 langgraph.json。
"""
from a2a_auto_wrapper import A2AClient
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from deepagents import create_deep_agent

from config import settings                    # 课案原文是 from config import setting

# CrewAI 分析师的 A2A 客户端（端口 2025）
crewai_client = A2AClient(base_url="http://localhost:2025/")


@tool
async def call_crewai_analyst(query: str) -> str:
    """需要数据分析或专业报告时调用 CrewAI 分析师。

    Args:
        query: 交给分析师的完整问题描述，包含待分析的数据

    Returns:
        分析师返回的分析报告文本
    """
    try:
        return await crewai_client.send_message(query)
    except Exception as e:
        return (
            f"调用 CrewAI 分析师失败：{e}。"
            "请确认 crewai_a2a_server.py 已在 2025 端口启动后重试。"
        )


agent = create_deep_agent(
    model=ChatOpenAI(
        model=settings.model_name,             # 课案原文 setting.MODEL_NAME
        api_key=settings.api_key,              # 课案原文 setting.API_KEY
        base_url=settings.base_url,            # 课案原文 setting.BASE_URL
        max_retries=3,
        timeout=60,
    ),
    system_prompt="你是总调度员。遇到数据分析任务时主动调用 call_crewai_analyst。",
    tools=[call_crewai_analyst],
)

graph = agent   # langgraph dev 加载入口（langgraph.json 中 deep_researcher 指向此处）
'''


COURSE_LANGGRAPH_JSON = '''{
  "python_version": "3.11",
  "dependencies": ["./"],
  "graphs": {
    "deep_researcher": "./deepagent_a2a.py:graph"
  },
  "env": ".env"
}'''


def build_local_langgraph_json() -> str:
    """把课案那份 langgraph.json 改写成「本机可直接用」的版本。

    差别只有两处（都为了让学员复制即用）：
        · python_version 取当前解释器真实版本，而不是课案的 3.11；
        · graphs 的路径用绝对路径指向本文件所在目录下的 deepagent_a2a.py。
    """
    this_dir = NB_DIR
    payload = {
        # platform.python_version() → "3.13.5"；langgraph.json 只认 "x.y"
        "python_version": ".".join(platform.python_version().split(".")[:2]),
        "dependencies": ["./"],
        "graphs": {
            # 真实部署时这个文件就叫 deepagent_a2a.py（就是上面那段课案代码）
            "deep_researcher": str(this_dir / "deepagent_a2a.py") + ":graph",
        },
        "env": ".env",
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)

### 3.3 把图暴露成 HTTP 服务：`langgraph dev`

课案原文的启动命令与三个参数/变量的含义：

| 内容 | 说明 |
|---|---|
| `langgraph-cli[inmem]` | CLI 本体 + inmem（内存版运行时），本地开发不需要起数据库 |
| `--port 2024` | 监听端口，和 CrewAI 的 2025 错开，这是本文档的约定 |
| `--no-browser` | 别自动开浏览器（服务器/无桌面环境必须加，否则卡住） |
| `LANGCHAIN_TRACING_V2=false` | 关掉 LangSmith 上报，避免没配 key 时刷一堆告警 |
| `$env:PYTHONUTF8="1"` | Windows 下让日志里的中文不乱码 |

跑起来之后：`http://localhost:2024/ok` 是健康检查、`/docs` 是自动生成的 API 文档、
`assistant_id = "deep_researcher"` 就是 `langgraph.json` 里 `graphs` 的键名。

In [ ]:
def print_dev_server_hint() -> None:
    print("【启动 DeepAgents 侧服务（课案原文，端口 2024）】")
    print('    pip install "langgraph-cli[inmem]"    # 本项目请用：uv add "langgraph-cli[inmem]"')
    print("    # Linux / macOS")
    print("    LANGCHAIN_TRACING_V2=false langgraph dev --port 2024 --no-browser")
    print("    # Windows PowerShell")
    print('    $env:PYTHONUTF8="1"; $env:LANGCHAIN_TRACING_V2="false"; '
          '.venv\\Scripts\\langgraph dev --port 2024 --no-browser')
    print()

### 3.4 调度员本体：真 client + 假 client

`build_crewai_client()` 是课案那行 `A2AClient(base_url="http://localhost:2025/")`；
`build_dispatcher()` 组装「一个 `@tool` + `create_deep_agent`」。为了**离线验证**工具里的
try/except 分支，还造了一个 `FakeA2AClient` —— 它故意抛 `ConnectionError`，正是「2025 端口
没起服务」时真实 `A2AClient` 会抛的东西。`probe_crewai_tool()` 绕开 LLM 直接 await 工具函数体，
把「工具内部逻辑」和「模型决策」拆开看。

In [ ]:
def build_crewai_client():
    """课案那行 `crewai_client = A2AClient(base_url="http://localhost:2025/")`。"""
    return A2AClient(base_url=CREWAI_ANALYST_URL)


async def call_crewai_analyst_impl(client, query: str) -> str:
    """call_crewai_analyst 的真实函数体 —— 课案那段 try/except 原样搬过来。

    单独抽成模块级函数，是为了让「降级演示」能直接 await 它本体，
    验证「2025 端口没起服务」时返回的是**一句人话**而不是异常。
    """
    try:
        # send_message 背后就是一次 A2A 的 message/send（HTTP POST JSON-RPC）
        return await client.send_message(query)
    except Exception as e:
        # 工具里吞掉异常、返回可读文本：模型看到这句话还能自己决定重试或换个说法
        return (
            f"调用 CrewAI 分析师失败：{e}。"
            "请确认 crewai_a2a_server.py 已在 2025 端口启动后重试。"
        )


def build_dispatcher(crewai_client):
    """组装总调度员：一个 @tool + create_deep_agent。

    注意工具定义写在函数内部，是为了能用外部传入的 crewai_client
    （课案是模块级写法，两者等价；本文件要同时演示「真 client」和「假 client」）。
    """
    from langchain_core.tools import tool
    from langchain_openai import ChatOpenAI

    @tool
    async def call_crewai_analyst(query: str) -> str:
        """需要数据分析或专业报告时调用 CrewAI 分析师。

        Args:
            query: 交给分析师的完整问题描述，包含待分析的数据

        Returns:
            分析师返回的分析报告文本
        """
        return await call_crewai_analyst_impl(crewai_client, query)

    # 调度员本体：模型用课案原文的 ChatOpenAI 三件套；
    # max_retries / timeout 也是课案原文，都跟「A2A 是网络调用」这件事有关。
    agent = create_deep_agent(
        model=ChatOpenAI(
            model=settings.model_name,
            api_key=settings.api_key,
            base_url=settings.base_url,
            max_retries=3,   # 网络抖动时自动重试 3 次，A2A 调用尤其需要
            timeout=60,      # 对方 LLM 可能想很久，60 秒够用
        ),
        # 提示词里点名「遇到数据分析任务时主动调用 call_crewai_analyst」，
        # 否则模型可能自己硬答、不去用那个工具
        system_prompt="你是总调度员。遇到数据分析任务时主动调用 call_crewai_analyst。",
        tools=[call_crewai_analyst],
    )
    return agent


class FakeA2AClient:
    """降级用的假 A2AClient：验证 call_crewai_analyst 的 try/except 分支。

    它故意抛 ConnectionError —— 这正是「2025 端口没起服务」时 A2AClient
    会抛的东西。学员能看到课案那段 try/except 到底在防什么。
    """

    def __init__(self, base_url: str) -> None:
        self.base_url = base_url

    async def send_message(self, query: str) -> str:
        raise ConnectionError(f"无法连接 {self.base_url}（Connection refused）")


async def probe_crewai_tool(client, label: str) -> None:
    """直接 await 工具函数体，看它拿到什么。

    绕开 LLM 直接调工具，是为了把「工具内部逻辑」和「模型决策」拆开看：
    A2A 调用失败时，工具返回的是一句话，而不是抛异常。
    调的就是 build_dispatcher 里那个 @tool 包着的同一个实现，不存在"抄一遍"。
    """
    print(f"[{label}] 直接调用工具函数（不经过模型）")
    result = await call_crewai_analyst_impl(
        client, "分析销售数据：1月100万 2月120万 3月90万，做趋势预测"
    )
    print("    返回：" + result)
    print()

### 3.5 探测 2024 端口（langgraph_sdk 真连一次）

`langgraph_sdk` 是**纯 HTTP 客户端**（本机已装），所以哪怕没装 `langgraph-cli`、没起服务，
也能跑这一段 —— 我们要的就是「连不上时给出人话指引」而不是甩一坨 traceback。

In [ ]:
async def probe_langgraph_server() -> None:
    """用 langgraph_sdk 探测 2024 端口。

    langgraph_sdk 是**纯 HTTP 客户端**（本机已装），所以哪怕没装
    langgraph-cli、没起服务，也能跑这一段 —— 我们要的就是「连不上时
    给出人话指引」而不是甩一坨 traceback。
    """
    print("【探测 DeepAgents 侧服务】http://localhost:2024")
    if not _HAS_LANGGRAPH_SDK:
        print("    langgraph_sdk 未安装，跳过。安装：uv add langgraph-sdk")
        print()
        return

    client = get_client(url=f"http://localhost:{DEEPAGENT_PORT}")
    try:
        # threads.create() 会真的发一次 HTTP 请求
        thread = await client.threads.create()
        print(f"    连接成功！已创建会话线程 thread_id = {thread['thread_id']}")
        print("    （05_a2a互相通信_jxsd.py 会用这个 thread_id 提交问题）")
    except Exception as exc:
        print(f"    连不上（{type(exc).__name__}）：{str(exc)[:120]}")
        print("    → 这是**预期结果**：DeepAgents 侧服务还没启动。")
        print("      按下面三步启动整套 A2A 链路后，本文件会走到「连接成功」分支：")
        print("        ① uv add a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai")
        print("        ② .venv\\Scripts\\python crewai_a2a_server.py          "
              "# CrewAI 分析师，:2025")
        print("        ③ langgraph dev --port 2024 --no-browser              "
              "# 总调度员，:2024")
    print()

### 3.6 主流程 `main()`：把上面全部串起来

本机是「缺 a2a_auto_wrapper」分支，所以走降级：用假 client 验证 try/except、再用
langgraph_sdk 真连一次 2024（连不上就给中文指引）。注意两处 `asyncio.run(...)` 都换成了
第 3.1 小节的 `run_async(...)`。

In [ ]:
def main() -> None:
    print("=" * 66)
    print("A2A 客户端：DeepAgents 总调度员（调 CrewAI 分析师 + 自身暴露为 A2A）")
    print("=" * 66)
    print()

    # ---------- 第 1 步：课案原文 ----------
    print("【课案原文】deepagent_a2a.py（58 行）：")
    for line in COURSE_DEEPAGENT_PY.splitlines():
        print("    " + line)
    print()

    print("【课案原文】langgraph.json（8 行，CrewAI/调度员的加载入口配置）：")
    for line in COURSE_LANGGRAPH_JSON.splitlines():
        print("    " + line)
    print()
    print("【本机可用版】langgraph.json（版本号与路径都换成动态生成）：")
    for line in build_local_langgraph_json().splitlines():
        print("    " + line)
    print()

    print_dev_server_hint()

    # ---------- 第 2 步：依赖情况 ----------
    print("【前置条件检查】")
    for err in _IMPORT_ERRORS:
        print(f"    - 缺失：{err}")
    print("      安装：uv add a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai")
    print('      另需：uv add "langgraph-cli[inmem]"    # 提供 langgraph dev 命令')
    print()

    # ---------- 第 3 步：A2AClient / create_deep_agent ----------
    # 依赖齐全就组装「真调度员」；缺 a2a_auto_wrapper 就换成两件降级的事。
    if _HAS_A2A_WRAPPER and _HAS_DEEPAGENTS:
        print("【依赖检查】a2a_auto_wrapper + deepagents 均已就绪，组装真实调度员。")
        client = build_crewai_client()
        agent = build_dispatcher(client)
        print(f"    调度员已就绪：{type(agent).__name__}，工具 = [call_crewai_analyst]")
        print(f"    它会通过 A2A 调用 {CREWAI_ANALYST_URL}（需先起 crewai_a2a_server.py）")
        print()
    else:
        # 降级分支刻意不「装样子」去调远程服务，只做两件有意义的事：
        #   ① 用假 client 直接 await 工具函数体 → 验证课案那段 try/except 真的能兜住失败；
        #   ② 交给第 4 步的 langgraph_sdk 去真连 :2024，连不上就给中文指引。
        print("【降级演示】a2a_auto_wrapper 未安装 → 无法 new 出真的 A2AClient。")
        print("            改成两件事：① 用假 client 验证工具里的 try/except 分支；")
        print("                      ② 用 langgraph_sdk（已装）真连一次 2024 端口看提示。")
        print()
        run_async(probe_crewai_tool(FakeA2AClient(CREWAI_ANALYST_URL),
                                     "假 A2AClient · 未起 2025 服务"))
        # deepagents 装了只能说明「这段代码语法可用」，不代表能跑通 ——
        # 真正跑起来还需要 :2025 有服务，所以这里只报告事实、不发起调用。
        if _HAS_DEEPAGENTS:
            print("【依赖检查】deepagents 已装 → 课案那段 create_deep_agent(...) 语法本身可用；")
            print("            只是 model 换成 settings 三件套、@tool 里的 client 换成假 client。")
            print("            真正跑起来需要 2025 端口有服务，本机不具备，故不发起调用。")
            print()

    # ---------- 第 4 步：探测 2024 端口 ----------
    run_async(probe_langgraph_server())

    print("=" * 66)
    print("小结：调度员 = 一个普通 DeepAgent + 一个「调远程 Agent」的 @tool。")
    print("      · A2AClient 把 A2A 的 HTTP JSON-RPC 藏进了 send_message()")
    print("      · create_deep_agent 出的图交给 langgraph.json 注册，")
    print("        langgraph dev --port 2024 一跑，它就同时成了 A2A 服务端")
    print("      · 三方串联（客户端 → 调度员 → 分析师）见 05_a2a互相通信_jxsd.py")
    print("=" * 66)


main()

### 预期输出

本机实测（`a2a_auto_wrapper` 缺、`deepagents` 已装 → 走降级分支；2024 端口无服务）：

```text
【课案原文】deepagent_a2a.py（58 行）：
    """DeepAgents A2A 服务：总调度员，可自主调用 CrewAI 分析师
    ...
    graph = agent   # langgraph dev 加载入口（langgraph.json 中 deep_researcher 指向此处）

【课案原文】langgraph.json（8 行，CrewAI/调度员的加载入口配置）：
    { ... "deep_researcher": "./deepagent_a2a.py:graph" ... }
【本机可用版】langgraph.json（版本号与路径都换成动态生成）：
    { "python_version": "3.13", "graphs": { "deep_researcher": "<本机路径>/deepagent_a2a.py:graph" } }

【前置条件检查】
    - 缺失：a2a_auto_wrapper：No module named 'a2a_auto_wrapper'
      安装：uv add a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai
      另需：uv add "langgraph-cli[inmem]"    # 提供 langgraph dev 命令

【降级演示】a2a_auto_wrapper 未安装 → 无法 new 出真的 A2AClient。
            改成两件事：① 用假 client 验证工具里的 try/except 分支；
                      ② 用 langgraph_sdk（已装）真连一次 2024 端口看提示。

[假 A2AClient · 未起 2025 服务] 直接调用工具函数（不经过模型）
    返回：调用 CrewAI 分析师失败：无法连接 http://localhost:2025/（Connection refused）。请确认 crewai_a2a_server.py 已在 2025 端口启动后重试。

【依赖检查】deepagents 已装 → 课案那段 create_deep_agent(...) 语法本身可用；
            只是 model 换成 settings 三件套、@tool 里的 client 换成假 client。
            真正跑起来需要 2025 端口有服务，本机不具备，故不发起调用。

【探测 DeepAgents 侧服务】http://localhost:2024
    连不上（ConnectError）：...   → 这是**预期结果**：DeepAgents 侧服务还没启动。 ...
```

> ⚠️ 连不上 2024 那行的**异常类型/文本每次可能不同**（`ConnectError` / `Connection refused` 等，
> 取决于本机网络栈与代理状态），但「连不上 → 给中文启动指引」这条路径是固定的；正文是实测值。

## 4. 完整版：互相通信 —— 三方串联

最后把 A2A 放回**真实链路**里看：普通 Python 脚本 →（langgraph_sdk / HTTP）→ DeepAgents 调度员
→（A2AClient / HTTP JSON-RPC）→ CrewAI 分析师，再逐层原路返回。A2A 管的是**后面那一跳**
（Agent ↔ Agent），前面那一跳是 LangGraph 平台自己的 API。

本节对应课案「互相通信」的 60 行 `a2a_client.py`，藏了 4 个工程要点：

| 写法 | 为什么 |
|---|---|
| `threads.create()` 先建线程 | LangGraph 里 thread 就是「一段会话」，复用才有上下文记忆 |
| `assistant_id="deep_researcher"` | 必须是 `langgraph.json → graphs` 里的**键名**，不是变量名 |
| `runs.wait(...)` | 服务端把整轮跑完才返回（非流式）；要边跑边看用 `runs.stream` |
| `2 ** attempt` 指数退避 | 第 1 次失败等 1s、第 2 次等 2s、第 3 次等 4s… 避免服务刚起就被连环打断 |
| `if attempt == max_attempts - 1: raise` | **最后一次不再吞异常**，否则重试耗尽后静默返回 None |

本机走降级：真连一次 2024（连不上给中文指引）+ 用假 client 把**指数退避逻辑**真的跑一遍。

### 4.1 依赖探测 + 常量

本节只需要 `langgraph_sdk`（纯 HTTP 客户端，本机已装）。`ASSISTANT_ID` 必须和
`langgraph.json` 里 `graphs` 的键名对得上，否则 `runs.wait` 会报 `assistant not found` ——
这是最常见的踩坑点。

In [ ]:
import asyncio
import time

_HAS_LANGGRAPH_SDK = False
_IMPORT_ERRORS: list[str] = []

try:
    from langgraph_sdk import get_client

    _HAS_LANGGRAPH_SDK = True
except ImportError as exc:
    _IMPORT_ERRORS.append(f"langgraph_sdk：{exc}")

DEEPAGENT_URL = "http://localhost:2024"
# langgraph.json 里 graphs 的键名 —— assistant_id 必须和它对上，
# 否则 runs.wait 会报「assistant not found」，这是最常见的踩坑点。
ASSISTANT_ID = "deep_researcher"

# 课案里那个问题（调度员看到「分析销售数据…趋势预测」就会去调 CrewAI 分析师）
SALES_QUESTION = "分析销售数据：1月100万 2月120万 3月90万，做趋势预测"

### 4.2 课案原文 `a2a_client.py`（60 行）+ 启动顺序

启动顺序（**顺序不能反**）：① `crewai_a2a_server.py`（:2025）→ ② `langgraph dev`（:2024）→
③ `a2a_client.py`。为什么不能反？因为 ② 的调度员在**模块导入时**就 new 出
`A2AClient(base_url="http://localhost:2025/")`；① 没起时，③ 一提问就会在工具里拿到 `Connection refused`。

In [ ]:
COURSE_CLIENT_PY = '''import asyncio

from langgraph_sdk import get_client


async def wait_with_retry(client, thread_id: str, max_attempts: int = 3) -> dict:
    """带指数退避重试的 runs.wait 封装。

    Args:
        client: langgraph_sdk 客户端实例
        thread_id: 会话线程 ID
        max_attempts: 最大尝试次数（含首次）

    Returns:
        运行完成后的最终状态字典
    """
    for attempt in range(max_attempts):
        try:
            return await client.runs.wait(
                thread_id,
                assistant_id="deep_researcher",
                input={
                    "messages": [
                        {
                            "role": "user",
                            "content": "分析销售数据：1月100万 2月120万 3月90万，做趋势预测",
                        }
                    ]
                },
            )
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            wait_seconds = 2 ** attempt
            print(f"运行失败（{e}），{wait_seconds} 秒后重试 ({attempt + 1}/{max_attempts})...")
            await asyncio.sleep(wait_seconds)


async def main() -> None:
    """创建会话线程并等待 DeepAgents 调度员执行完成，打印最终回复。"""
    client = get_client(url="http://localhost:2024")
    thread = await client.threads.create()
    result = await wait_with_retry(client, thread["thread_id"])
    print(result["messages"][-1]["content"])


if __name__ == "__main__":
    asyncio.run(main())
'''


def print_startup_guide() -> None:
    """把课案的启动顺序与命令原样打出来 —— 这是本节最容易做错的一步。"""
    print("【课案原文：启动顺序（不能反）+ 运行命令】")
    print("    ① .venv\\Scripts\\python crewai_a2a_server.py          # CrewAI 分析师 :2025")
    print("    ② .venv\\Scripts\\langgraph dev --port 2024 --no-browser  # 总调度员 :2024")
    print('    ③ $env:PYTHONUTF8="1"; .venv\\Scripts\\python a2a_client.py')
    print('    # PYTHONUTF8="1"：让 stdio 走 UTF-8，避免 emoji/中文在 GBK 控制台炸掉')
    print()

### 4.3 课案原函数 `wait_with_retry`（一行没改）

这是课案里最值得抄的 30 行。两个注释要点：`assistant_id` 必须是 `langgraph.json` 里的**键名**
（不是 Python 变量名）；最后一次重试**不再吞异常**（`if attempt == max_attempts - 1: raise`），
否则重试耗尽后函数会静默返回 None，调用方拿到 None 才发现出事。

In [ ]:
async def wait_with_retry(client, thread_id: str, max_attempts: int = 3) -> dict:
    """带指数退避重试的 runs.wait 封装。

    Args:
        client: langgraph_sdk 客户端实例
        thread_id: 会话线程 ID
        max_attempts: 最大尝试次数（含首次）

    Returns:
        运行完成后的最终状态字典
    """
    for attempt in range(max_attempts):
        try:
            # runs.wait：服务端把**整轮**跑完才返回（非流式）。
            # 注意 assistant_id 必须是 langgraph.json → graphs 里的**键名**，
            # 不是 Python 变量名 —— 写错就拿不到图，报 assistant not found。
            return await client.runs.wait(
                thread_id,
                assistant_id=ASSISTANT_ID,   # 课案是字面量 "deep_researcher"，这里提成常量
                input={
                    "messages": [
                        {
                            "role": "user",
                            # 这句就是「让调度员自己决定要不要去调 CrewAI 分析师」的输入
                            "content": SALES_QUESTION,
                        }
                    ]
                },
            )
        except Exception as e:
            # 最后一次不再吞异常（原因见文件头第四节）——否则重试耗尽后
            # 函数会静默返回 None，调用方拿到 None 才发现出事，排查成本更高
            if attempt == max_attempts - 1:
                raise
            # 指数退避：第 1 次失败等 1s、第 2 次等 2s、第 3 次等 4s…
            # 目的：服务刚起、还在预热时，别被客户端连环打断
            wait_seconds = 2 ** attempt
            print(f"运行失败（{e}），{wait_seconds} 秒后重试 ({attempt + 1}/{max_attempts})...")
            await asyncio.sleep(wait_seconds)

### 4.4 假 client + 退避演示 + 真探测

`FlakyClient` 失败 N 次后成功，`.runs` / `.threads` 属性模仿 `langgraph_sdk` 的形状，
所以丢给 `wait_with_retry` 的**函数一行都不用改**。`demo_retry_with_fake_client` 用
`fail_times=2` 跑「失败两次、第三次成功、退避 1s→2s」；`demo_timeout_with_real_sdk` 真连一次 2024。

In [ ]:
class FlakyClient:
    """失败 N 次后成功的假 client —— 专门用来验证 wait_with_retry 的重试分支。

    `.runs` 属性模仿 langgraph_sdk 的形状（client.runs.wait / client.threads.create），
    所以丢给 wait_with_retry 的那个函数**一行都不用改**。
    """

    class _Runs:
        def __init__(self, fail_times: int) -> None:
            self.fail_times = fail_times
            self.calls = 0

        async def wait(self, thread_id, assistant_id, input):  # noqa: A002 - 对齐官方签名
            self.calls += 1
            if self.calls <= self.fail_times:
                # 模拟「服务刚起、还在预热」时的连接失败
                raise ConnectionError(
                    f"Connection refused to {DEEPAGENT_URL}（{assistant_id} 尚未就绪）"
                )
            # 成功时返回的字典结构对齐课案：result["messages"][-1]["content"]
            question = input["messages"][-1]["content"]
            return {
                "messages": [
                    {"role": "user", "content": question},
                    {
                        "role": "assistant",
                        "content": (
                            "【模拟回复（假 client，非真实模型输出）】\n"
                            "已完成调度：call_crewai_analyst → A2A → :2025 数据分析师。\n"
                            "结论：1→2 月 +20%，2→3 月 -25%，属单峰波动；"
                            "建议排查 3 月环比下滑原因，Q2 目标按 1 月水平 +10% 设定。"
                        ),
                    },
                ]
            }

    class _Threads:
        def __init__(self) -> None:
            self.created = 0

        async def create(self):
            self.created += 1
            return {"thread_id": f"thread-flaky-{self.created}"}

    def __init__(self, fail_times: int) -> None:
        self.runs = FlakyClient._Runs(fail_times)
        self.threads = FlakyClient._Threads()


async def demo_retry_with_fake_client() -> None:
    """用假 client 跑 main() 的同一套流程，证明重试逻辑真的生效。"""
    print("【降级演示】用假 client 跑通 wait_with_retry 的指数退避分支")
    print("            （2024 端口没服务，但重试逻辑本身可以离线验证）")
    client = FlakyClient(fail_times=2)   # 前两次失败，第三次成功
    thread = await client.threads.create()
    print(f"    已创建线程：thread_id = {thread['thread_id']}")

    started = time.perf_counter()
    result = await wait_with_retry(client, thread["thread_id"])   # ← 课案原函数
    elapsed = time.perf_counter() - started
    print(f"    最终回复（result['messages'][-1]['content']）：")
    for line in result["messages"][-1]["content"].splitlines():
        print("        " + line)
    print(f"    共尝试 {client.runs.calls} 次，退避累计约 {elapsed:.1f} 秒"
          f"（理论：1s + 2s = 3s）")
    print()


async def demo_timeout_with_real_sdk() -> None:
    """真连一次 2024 端口：连不上就给出中文指引（不抛 traceback）。"""
    print("【探测真实服务】http://localhost:2024")
    if not _HAS_LANGGRAPH_SDK:
        print("    langgraph_sdk 未安装，跳过。安装：uv add langgraph-sdk")
        print()
        return

    client = get_client(url=DEEPAGENT_URL)
    try:
        thread = await client.threads.create()
        print(f"    连接成功！thread_id = {thread['thread_id']}")
        print("    正在提交问题并等待调度员执行（可能要几十秒，期间它会去调 :2025）…")
        result = await wait_with_retry(client, thread["thread_id"])
        # 课案最后一行：把最终回复打出来
        print(result["messages"][-1]["content"])
    except Exception as exc:
        # 连不上是**预期结果**（本机没起服务），所以给出可照做的启动清单而不是 traceback
        print(f"    连不上（{type(exc).__name__}）：{str(exc)[:140]}")
        print("    → 预期结果：本机没装依赖、也没起服务。完整链路需要：")
        print("      ① uv add a2a_auto_wrapper langgraph-api langgraph-sdk deepagents crewai")
        print('      ② uv add "langgraph-cli[inmem]"                         # 提供 langgraph dev')
        print("      ③ 写两个服务文件：crewai_a2a_server.py / deepagent_a2a.py")
        print("         （内容见 03_a2a服务端_jxsd.py、04_a2a客户端_jxsd.py 里的课案原文）")
        print("      ④ 按上面的启动顺序跑 ①②③")
        print("    上面的退避演示已经把客户端逻辑验证完了，连上服务后本文件会走成功分支。")
    print()

### 4.5 主流程 `main()`：退避演示 + 真探测

同样两处 `asyncio.run(...)` 换成 `run_async(...)`。结构固定：先摊开课案原文 → 前置条件 →
先验证客户端逻辑（降级）→ 最后真连一次 2024。

In [ ]:
def main() -> None:
    # 结构固定：先摊开课案原文 → 再看前置条件 → 先验证客户端逻辑（降级）
    # → 最后真连一次 2024 端口。无论连不连得上，本文件都能跑完并给出结论。
    print("=" * 66)
    print("A2A 互相通信：a2a_client → DeepAgents 调度员 → CrewAI 分析师")
    print("=" * 66)
    print()

    # 逐行打印课案那 60 行客户端，方便对照后面的真实调用
    print("【课案原文】a2a_client.py（60 行）：")
    for line in COURSE_CLIENT_PY.splitlines():
        print("    " + line)
    print()

    print_startup_guide()

    print("【前置条件检查】")
    if _IMPORT_ERRORS:
        for err in _IMPORT_ERRORS:
            print(f"    - 缺失：{err}")
        print("      安装：uv add langgraph-sdk langgraph-api")
    else:
        # langgraph_sdk 是纯 HTTP 客户端，不依赖服务端存在就能 import
        print("    - langgraph_sdk：已就绪（纯 HTTP 客户端，能独立探测服务是否在线）")
    print()

    # ---------- 降级：先把客户端逻辑验证掉 ----------
    run_async(demo_retry_with_fake_client())

    # ---------- 真实：连一次 2024 ----------
    run_async(demo_timeout_with_real_sdk())

    print("=" * 66)
    print("小结：A2A 的完整链路 = 三次跳转、两种传输 ——")
    print("    a2a_client ──langgraph_sdk(HTTP)──► :2024 调度员")
    print("    调度员     ──A2AClient(HTTP JSON-RPC)──► :2025 CrewAI 分析师")
    print("    分析师报告 ──────────────────────────► 逐层原路返回")
    print("    每一跳都是「标准协议 + 普通 HTTP」，所以框架不同也能拼起来。")
    print("=" * 66)


main()

### 预期输出

本机实测（`langgraph_sdk` 已装、2024 端口无服务 → 走降级）：

```text
【课案原文】a2a_client.py（60 行）：
    import asyncio
    from langgraph_sdk import get_client
    async def wait_with_retry(...):
        ...
    if __name__ == "__main__":
        asyncio.run(main())

【课案原文：启动顺序（不能反）+ 运行命令】
    ① .venv\Scripts\python crewai_a2a_server.py          # CrewAI 分析师 :2025
    ② .venv\Scripts\langgraph dev --port 2024 --no-browser  # 总调度员 :2024
    ③ $env:PYTHONUTF8="1"; .venv\Scripts\python a2a_client.py

【前置条件检查】
    - langgraph_sdk：已就绪（纯 HTTP 客户端，能独立探测服务是否在线）

【降级演示】用假 client 跑通 wait_with_retry 的指数退避分支
            （2024 端口没服务，但重试逻辑本身可以离线验证）
    已创建线程：thread_id = thread-flaky-1
    运行失败（Connection refused to http://localhost:2024（deep_researcher 尚未就绪）），1 秒后重试 (1/3)...
    运行失败（Connection refused to http://localhost:2024（deep_researcher 尚未就绪）），2 秒后重试 (2/3)...
    最终回复（result['messages'][-1]['content']）：
        【模拟回复（假 client，非真实模型输出）】
        已完成调度：call_crewai_analyst → A2A → :2025 数据分析师。
        结论：1→2 月 +20%，2→3 月 -25%，属单峰波动；建议排查 3 月环比下滑原因，Q2 目标按 1 月水平 +10% 设定。
    共尝试 3 次，退避累计约 3.0 秒（理论：1s + 2s = 3s）

【探测真实服务】http://localhost:2024
    连不上（ConnectError）：... → 预期结果：本机没装依赖、也没起服务。完整链路需要： ...
```

> ⚠️ 两处是**每次运行可能不同**的实测值：①「退避累计约 3.0 秒」里的 `3.0` 是计时结果，
> 实际在 3.0 附近浮动；② 真探测 2024 的异常类型/文本取决于本机网络栈。结构（失败两次→成功、
> 退避 1s→2s）是固定的。

## 小结

- **A2A = Agent ↔ Agent**，管的是「不同框架、不同厂商写的智能体互相下单/交货」这条边；
  对照记：ACP = 编辑器 ↔ Agent，MCP = Agent ↔ 工具，三者互不替代。
- **传输层是 JSON-RPC 2.0 over HTTP**：报文格式和 ACP 那套完全一样（有 id 是请求、无 id 是通知），
  只是从「stdio 一行一个 JSON」换成「HTTP POST 一个 JSON」。
- **A2A 服务端就两件事**：① 在 `/.well-known/agent.json` 挂一张「名片」（我是谁、会什么、在哪）；
  ② 在同一地址收 `message/send`（GET 发现 + POST 调用）。`serve_crewai_agent(analyst, port=2025)`
  就是把这两件事自动配好。
- **调度员 = 普通 DeepAgent + 一个「调远程 Agent」的 `@tool`**：`A2AClient` 把 HTTP JSON-RPC 藏进
  `send_message()`，模型看到的只是「一个叫 call_crewai_analyst 的工具」。
- **三方串联 = 三次跳转、两种传输**：`a2a_client ──langgraph_sdk(HTTP)──► :2024 调度员
  ──A2AClient(JSON-RPC)──► :2025 分析师`，每一跳都是「标准协议 + 普通 HTTP」，框架不同也能拼起来。

## 常见坑

1. **编号错位**：`02_a2a服务端.py` 和 `03_a2a服务端_jxsd.py` 才是「服务端」的原版/完整版配对，
   `03_a2a客户端.py` 配 `04_a2a客户端_jxsd.py` —— 别按文件名里的数字硬配对。
2. **`parts` 是内容块数组，不是字符串**：收响应时要 `artifacts[0].parts[0].text` 一路抠进去；
  拼用户输入时也要过滤 `kind == "text"` 再 `join`，不能直接把 `parts` 当字符串。
3. **响应里 `id` 必须回同 id**：这是 JSON-RPC 的配对规则（请求发几号，响应就回几号），
  方法名写错会拿到标准错误码 `-32601`。
4. **端口别撞**：课案固定 2024（调度员）/ 2025（CrewAI 分析师）；演示降级版用
  `ThreadingHTTPServer(("127.0.0.1", 0), ...)` 让 OS 分空闲端口，比写死更抗并发。
5. **Jupyter 内核里别直接 `asyncio.run()`**：内核本身就在事件循环里，直接跑会抛
  `RuntimeError: asyncio.run() cannot be called from a running event loop`。本课用
  `run_async()`（另起线程再 `asyncio.run`）兜住（见第 3.1 节）。
6. **`assistant_id` 必须是 `langgraph.json → graphs` 的键名**，不是 Python 变量名 —— 写错会报
  `assistant not found`。
7. **工具里不要往外抛异常**：`call_crewai_analyst` 里 try/except 返回一句人话，模型才能决定重试；
  抛了会直接变成 Agent 运行失败。

## 官方链接

- A2A 协议官方规范：<https://a2a-protocol.org>
- A2A 协议 GitHub：<https://github.com/a2aproject/A2A>
- Agent Card（.well-known/agent-card.json）规范：<https://a2a-protocol.org/latest/specification/#agent-card>
- AutoA2A（a2a_auto_wrapper 的上游，serve_crewai_agent）：<https://github.com/ag2ai/autoa2a>
- LangGraph 平台（langgraph dev / assistant_id）：<https://docs.langchain.com/oss/python/langgraph/cloud>